# CS 691 · Module 10 · Checking a language model’s claims

Break a generated answer into individual claims and classify each claim using records for the correct item and time.

**Before you begin.** You can retrieve the record IDs for a question and follow a link such as C1 → pickup_at → West Desk.

C1’s pickup record says North Desk and C2’s says South Desk. A message about C1 cites the C2 record. What would you inspect before trusting the message?

<details><summary>Reveal the answer</summary>

Match the item, relationship and location in the cited record to the message. A real pickup record can still be the wrong source if it describes C2.

</details>

## A fluent sentence can contain two claims

A large language model (LLM) generates text by repeatedly predicting a next piece of text from the pieces that precede it. The input can include a question and retrieved records. Grounding means tying what the answer says to evidence. To check a sentence, first find the individual statements a reader could verify.

A prompt is the text supplied to a model. The code assembles a question, instructions and two retrieved records into a prompt. Suppose the model replies, “Pick up C1 at West Desk before noon.” Circle the two claims in that draft. Which record, if any, supports each one?

**Predict before running the cell.** In the sample reply, which claim follows from the prompt’s records and which would require more evidence?

In [ ]:
question = "Where and by when may I collect C1?"
records = {"D8": "C1 pickup is at West Desk.",
           "H3": "C2 pickup ends at noon."}
lines = ["Answer from the records. Cite IDs. Say what is unknown.", question]
for record_id, passage in records.items():
    lines.append(f"[{record_id}] {passage}")
print("\n".join(lines))

<details><summary>Reveal output and explanation</summary>

```text
Answer from the records. Cite IDs. Say what is unknown.
Where and by when may I collect C1?
[D8] C1 pickup is at West Desk.
[H3] C2 pickup ends at noon.
```

D8 gives C1’s pickup desk. H3 gives a deadline for C2, so the time for C1 remains unknown. A supported reply could say, “Collect C1 at West Desk [D8]. I do not have a pickup deadline for C1.”

</details>

## Find support at the level of one claim

A claim is supported when the evidence establishes what the claim says. Compare the item, relation and value together. A citation is a reference to a source. It helps the reader check your work only when the named source actually supports that claim.

In this snapshot D8 gives C1’s pickup location. H3 gives a deadline for C2. Citing H3 for C1 would quietly switch items.

**Predict before running the cell.** Would searching only for the word “noon” catch the item mix-up?

In [ ]:
records = {"D8": ("C1", "pickup_at", "West Desk"),
           "H3": ("C2", "pickup_deadline", "noon")}
claims = [("C1", "pickup_at", "West Desk"),
          ("C1", "pickup_deadline", "noon")]
for claim in claims:
    ids = [key for key, fact in records.items() if fact == claim]
    print(claim[1], ", ".join(ids) or "no matching source")

<details><summary>Reveal output and explanation</summary>

```text
pickup_at D8
pickup_deadline no matching source
```

No. H3 contains “noon” but its item is C2. A claim about C1 needs evidence about C1.

</details>

## Tell a contradiction from a gap

A contradicted claim conflicts with something the evidence establishes. An unknown claim has insufficient evidence either way. Suppose our fictional desk assigns each camera exactly one pickup location for a given day, and D8 is current for today. Under that assumption, West Desk rules out East Desk for C1 today.

For C2, today’s desk is missing. The same candidate answer “East Desk” has a different status because the available evidence differs.

**Predict before running the cell.** What changes if several desks can serve C1 and D8 lists just one of them?

In [ ]:
today = {"C1": "West Desk"}
for item, candidate in [("C1", "West Desk"), ("C1", "East Desk"),
                        ("C2", "East Desk")]:
    if item not in today:
        status = "unknown"
    elif today[item] == candidate:
        status = "supported"
    else:
        status = "contradicted"
    print(item, candidate, status)

<details><summary>Reveal output and explanation</summary>

```text
C1 West Desk supported
C1 East Desk contradicted
C2 East Desk unknown
```

West Desk remains supported. East Desk becomes unknown unless another record excludes it. One listed option no longer rules out other options.

</details>

## Make time part of the claim

A useful source must cover the time in the question. A note can be accurate about Monday and irrelevant to Tuesday. When a draft omits that distinction, add the date to both the claim and the lookup.

Monday’s D8 and Tuesday’s D10 give different pickup locations for C1. The word “where” alone does not choose between them.

**Predict before running the cell.** A user forwards Monday’s answer without the date. Why should Tuesday’s desk avoid reusing it?

In [ ]:
desks = {("C1", "Monday"): ("West Desk", "D8"),
         ("C1", "Tuesday"): ("East Desk", "D10")}
item, day = "C1", "Tuesday"
location, source = desks[(item, day)]
print(f"On {day}, collect {item} at {location} [{source}].")

<details><summary>Reveal output and explanation</summary>

```text
On Tuesday, collect C1 at East Desk [D10].
```

Monday’s record establishes the location only for Monday. Tuesday’s record gives East Desk. Omitting the day hides which claim the citation supports.

</details>

## Check the camera in each proposed claim

Each proposed claim is an ordered group of three values. The loop tests it against every stored fact and collects IDs only when all three values agree. Joining those IDs shows the supporting sources, while an empty match yields the fallback message.

Replace this line:
           "H3": ("C2", "pickup_deadline", "noon")}
with:
           "H3": ("C1", "pickup_deadline", "noon")}
What will print?

In [ ]:
records = {"D8": ("C1", "pickup_at", "West Desk"),
           "H3": ("C2", "pickup_deadline", "noon")}
claims = [("C1", "pickup_at", "West Desk"),
          ("C1", "pickup_deadline", "noon")]
for claim in claims:
    ids = [key for key, fact in records.items() if fact == claim]
    print(claim[1], ", ".join(ids) or "no matching source")

<details><summary>Worked answer after your change</summary>

pickup_at D8
pickup_deadline H3

Changing the camera in H3 makes the item, relationship and value all agree with the deadline claim.

</details>

## Exercise 1

Split “Mia may collect T2 at Room 4 tomorrow without showing ID” into claims. Which one needs policy evidence?

<details><summary>Hint</summary>

Separate permission, place and ID requirements. Keep the day attached to each claim.

</details>

<details><summary>Worked answer</summary>

Check Mia’s permission to collect T2 tomorrow, T2’s pickup room tomorrow, and whether the relevant policy permits collection without ID. The last claim requires an applicable ID policy, and the first may also require checking eligibility rules.

</details>

## Exercise 2

There is exactly one current due date per loan. L2 lists Friday for Ava’s C1 loan. L3 lists Thursday for Ben’s C2 loan. Classify “Ava’s C1 is due Thursday” and “Ava owes a late fee.”

<details><summary>Hint</summary>

What does L2 exclude? What do neither of these records discuss?

</details>

<details><summary>Worked answer</summary>

The Thursday due-date claim is contradicted by Ava’s Friday date in L2. The fee claim is unknown. Neither record gives a fee assessment, and L3 belongs to a different loan.

</details>

## Exercise 3

A correct answer says “Room 4” but cites an old record for Room 1. Would a check that compares only the answer text to an answer key catch the defect? Design a stronger check.

<details><summary>Hint</summary>

Try to reproduce the answer using only the cited record, the item and the date.

</details>

<details><summary>Worked answer</summary>

A text-only check could accept Room 4. Fetch the cited ID and compare its item, relation, date and value with the claim. The old Room 1 record fails that check. Supply the current Room 4 record if available.

</details>

You checked a sentence against a record of what is true now. What changes when a model proposes several actions, each of which can change what is true next?

Ava verifies that C1 is available now. A proposed sequence reserves it for Bo and then promises Ava she can reserve it too. Why check the second step again?

<details><summary>Reveal the answer</summary>

The first action can change availability. A claim that was supported before Bo’s reservation may no longer hold when Ava’s step is reached.

</details>